## Lexers

In [12]:
import re

def load_patterns(filename="patterns.txt"):
    patterns = []
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if line.startswith('#') or not line:
                continue
            regex_str, kind = map(str.strip, line.split("\t", 1))
            patterns.append((re.compile(regex_str), kind))
    return patterns

def load_keyword(filename="keywords.txt"):
    keyword = {}
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if line.startswith('#') or not line:
                continue
            if ':' in line:
                word, kind = map(str.strip, line.split(":", 1))
                keyword[word] = kind
    return keyword

def load_source(filename="snippets/Case1ae.SRC"):
    source = ""
    with open(filename) as f:
        source = f.read()
    return source

In [40]:
class Token:
    def __init__(self, kind: str, value: str):
        self.kind = kind
        self.value = value

    def to_string(self):
        if self.kind in {"IDENTIFIER", "STRING", "NUMBER"}:
            return f"{self.kind.upper()}({self.value})"
        else: return f"{self.kind.upper()}"

    def debug(self):
        pass
    #     if self.kind in {"IDENTIFIER", "STRING", "NUMBER"}:
    #         print(f"{self.kind.lower()}({self.value})")
    #     else:
    #         print(f"{self.kind.lower()}()")

class Lexer:
    def __init__(self, source, patters, keywords):
        self.source = source
        self.pos = 0
        self.tokens = []
        self.patterns = patters
        self.keywords = keywords
        
    def remainder(self):
        return self.source[self.pos:]

    def advance_n(self, n):
        self.pos += n

    def push(self, token):
        self.tokens.append(token)

    def at_eof(self):
        return self.pos >= len(self.source)

    def tokenize(self):
        while not self.at_eof():
            matched = False
            for regex, kind in self.patterns:
                match = regex.match(self.remainder())
                if match:
                    val = match.group(0)
                    if kind == "COMMENT":
                        self.advance_n(len(val))
                    elif kind == "SKIP":
                        self.push(Token(kind, None))
                        self.advance_n(len(val))
                    elif kind == "IDENTIFIER":
                        actual_kind = self.keywords.get(val, "IDENTIFIER")
                        self.push(Token(actual_kind, val))
                        self.advance_n(len(val))
                    else:
                        self.push(Token(kind, val))
                        self.advance_n(len(val))
                    matched = True
                    break
            if not matched:
                raise Exception(f"Unrecognized token near: {self.remainder()[:30]}")
        self.push(Token("EOF", "EOF"))
        return self.tokens


In [42]:
import os

source = """
let x = 42.34; if (x >= 10) { x += 1; } // comment
"""
# source = load_source("snippets/DemoDashboard1.SRC")
patterns, keywords = load_patterns(), load_keyword()
root_dest = "tokens"
for root, _, files in os.walk("snippets"):
    for file in files:
        if file in [
            "CreateVariablesTest copy.SRC", 
            "CreateVariablesTest.SRC",
            "RecursionCheckTest copy.SRC",
            "RecursionCheckTest.SRC",
            "ServiceCtrlLogic.SRC",
            "TempCmn.SRC",
            "TempCnf.SRC",
            "Template.SRC",
            "Template.SRC_.SRC",
            "Wizard.SRC"
        ]: continue
        if file.endswith(".SRC"):
            src_path = os.path.join(root, file)
            dest_path = os.path.join(root_dest, file.replace(".SRC", ".TOK"))
            print(src_path, dest_path)
            with open(src_path, "r", encoding="utf-8") as f:
                source = f.read()
            lexer = Lexer(source, patterns, keywords)
            tokens = lexer.tokenize()
            with open(dest_path, "w", encoding="utf-8") as f:
                for t in tokens:
                    f.write(f"{t.to_string()}\n")


snippets\AppRoot.SRC tokens\AppRoot.TOK
snippets\BlkPage.SRC tokens\BlkPage.TOK
snippets\BuildTempModule.SRC tokens\BuildTempModule.TOK
snippets\CallCase5.SRC tokens\CallCase5.TOK
snippets\Case1.SRC tokens\Case1.TOK
snippets\Case1ae.SRC tokens\Case1ae.TOK
snippets\Case1be.SRC tokens\Case1be.TOK
snippets\Case1ce.SRC tokens\Case1ce.TOK
snippets\Case1de.SRC tokens\Case1de.TOK
snippets\Case1ee.SRC tokens\Case1ee.TOK
snippets\Case1fe.SRC tokens\Case1fe.TOK
snippets\Case1ge.SRC tokens\Case1ge.TOK
snippets\Case1he.SRC tokens\Case1he.TOK
snippets\Case1ie.SRC tokens\Case1ie.TOK
snippets\Case1je.SRC tokens\Case1je.TOK
snippets\Case1ke.SRC tokens\Case1ke.TOK
snippets\Case1le.SRC tokens\Case1le.TOK
snippets\Case2ae.SRC tokens\Case2ae.TOK
snippets\Case2be.SRC tokens\Case2be.TOK
snippets\Case3ae.SRC tokens\Case3ae.TOK
snippets\Case3be.SRC tokens\Case3be.TOK
snippets\Case3ce.SRC tokens\Case3ce.TOK
snippets\Case3de.SRC tokens\Case3de.TOK
snippets\Case3ee.SRC tokens\Case3ee.TOK
snippets\Case4.SRC token